In [101]:
#coding=utf-8
# ------------------------------------------------------------------------------------------#
# 
#-------------------------------------------------------------------------------------------#
from netCDF4 import Dataset as ncfile
from netCDF4 import num2date, date2num
import xarray as xr
from datetime import datetime, timedelta
import os, fnmatch, glob
import numpy as np
import numpy.ma as ma
import pandas as pd
import math
import matplotlib.dates as dates
from pyproj import Proj, transform
import scipy
from scipy import spatial
import cartopy.crs as ccrs
import cmocean
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [126]:
# Find nearest neighbor point index
def do_kdtree(combined_x_y_arrays,points):
        mytree = scipy.spatial.cKDTree(combined_x_y_arrays)
        dist, indexes = mytree.query(points)
        return indexes
    
# projection of the grid -> era5 on to modis       
def remapfield(MODvar,idx_pos_ij):
    MODprojvar=np.zeros(np.shape(idx_pos_ij))
    for i in range(np.shape(idx_pos_ij)[0]):
        for j in range(np.shape(idx_pos_ij)[1]):
            MODprojvar[i][j]=MODvar[idx_pos_ij[i][j]]
    return MODprojvar

class SpatialEvaluation:
    
    def __init__(self,modeleval):
        self.model=modeleval
    
    def defineGridTransforms(self):
        
        #MODIS grid (same as AROME Arctic)      
        modisin='/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/observations/remotesensing/aa-modis-2024.nc'
        AROMEin='/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/MET_AROMEArctic/AROME_ARCTIC/2024/04/04/arome_arctic_det_2_5km_20240404T00Z.nc'
        #
        modisdataset = ncfile(modisin, 'r')     
        aromedataset = ncfile(AROMEin, 'r') # not sufficient information in modis file
        modis_lons = aromedataset.variables['longitude'][:,:]
        modis_lats = aromedataset.variables['latitude'][:,:]
        modis_x = aromedataset.variables['x'][:]
        modis_y = aromedataset.variables['y'][:]
        modis_times = modisdataset.variables['time'][:]
        t_unit = modisdataset.variables['time'].units
        modis_nctimes=[]; modis_nctimes.append(num2date(modis_times,units = t_unit,calendar = 'proleptic_gregorian'))
        modis_nctimes=np.squeeze(modis_nctimes)
        modis_nctimenum = np.squeeze(dates.date2num(modis_nctimes))        
        #
        modis_proj = Proj("+proj=lcc +lat_1=77.5 +lat_2=77.5 +lat_0=77.5 +lon_0=-25 +R=6371000")          
        modisdataset.close()
        aromedataset.close()
            
        if self.model!='MET-AROMEArctic' and self.model!='MF-AROME' and self.model!='MF-ARPEGE':
                   
            if self.model=='ECMWF-IFS': 
                # ----- IFS Grid information
                fc_data_path='/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/ECMWF-IFS/'
                MODin = fc_data_path+'Arctic.ECMWF_extracted_20240401T00Z.nc'
                #
                MODdataset = ncfile(MODin, 'r')
                MOD_lons = MODdataset.variables['lon'][:]
                MOD_lats = MODdataset.variables['lat'][:]
                MOD_times = MODdataset.variables['time'][:]
                t_unit = MODdataset.variables['time'].units
                t_cal = MODdataset.variables['time'].calendar
                MOD_nctimes=[]; MOD_nctimes.append(num2date(MOD_times,units = t_unit,calendar = t_cal))
                MOD_nctimenum = np.squeeze(dates.date2num(MOD_nctimes))
                #
                MOD_proj = Proj(init = 'epsg:4326')
                MODdataset.close()

            if self.model=='ECMWF-AIFS':
                fc_data_path='/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/ECMWF-AIFS/'
                MODin = fc_data_path+'Arctic.ECMWF_AIFS_extracted_20240401T00Z.nc'
                #
                MODdataset = ncfile(MODin, 'r')
                MOD_lons = MODdataset.variables['lon'][:]
                MOD_lats = MODdataset.variables['lat'][:]
                MOD_times = MODdataset.variables['time'][:]
                t_unit = MODdataset.variables['time'].units
                t_cal = MODdataset.variables['time'].calendar
                MOD_nctimes=[]; MOD_nctimes.append(num2date(MOD_times,units = t_unit,calendar = t_cal))
                MOD_nctimenum = np.squeeze(dates.date2num(MOD_nctimes))
                #
                MOD_proj = Proj(init = 'epsg:4326')
                MODdataset.close()
            
            if self.model=='DWD-ICON':
                fc_data_path='/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/DWD-ICON/'
                MODin = fc_data_path+'region.icon_global_2024041012_T_ICE.nc'
                #
                MODdataset = ncfile(MODin, 'r')
                MOD_lons = MODdataset.variables['lon'][:]
                MOD_lats = MODdataset.variables['lat'][:]
                MOD_times = MODdataset.variables['time'][:]
                t_unit = MODdataset.variables['time'].units
                t_cal = MODdataset.variables['time'].calendar
                MOD_nctimes=[]; MOD_nctimes.append(num2date(MOD_times,units = t_unit,calendar = t_cal))
                MOD_nctimenum = np.squeeze(dates.date2num(MOD_nctimes))
                #
                MOD_proj = Proj(init = 'epsg:4326')
                MODdataset.close()
            
            
            MOD_llon, MOD_llat = np.meshgrid(MOD_lons,MOD_lats)
            modis_xx, modis_yy   = np.meshgrid(modis_x, modis_y)

            # Project all on MODIS grid
            MOD_xx,MOD_yy = transform(MOD_proj, modis_proj, MOD_llon, MOD_llat)

            # Vectorize and concatenate the x,y coordinates
            modis_xx_yy = np.dstack([modis_xx.ravel(), modis_yy.ravel()])[0]
            MOD_xx_yy  = np.dstack([MOD_xx.ravel(), MOD_yy.ravel()])[0]

            #idx_pos = do_kdtree(modis_xx_yy, era5_xx_yy)     #Gives a field with dimensions of era5_xx_yy
            idx_pos = do_kdtree( MOD_xx_yy, modis_xx_yy)     #Gives a field with dimensions of modis_xx_yy

            idx_pos_ij = idx_pos.reshape(np.shape(modis_xx))  #Gives a field with dimensions of modis_xx    

            self.idx_pos_ij = idx_pos_ij
        
        self.modis_lons=modis_lons
        self.modis_lats=modis_lats
        self.modis_nctimes=modis_nctimes
        self.modis_nctimenum=modis_nctimenum
        
        
    def ModelTransform(self,filein,timestep,variable):
        
        
        if self.model=='MET-AROMEArctic':
            fc_data_path='/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/MET-AROMEArctic/'
            MODin = fc_data_path+filein
            #
            #filein = f'MET-AROMEArctic/AROME_ARCTIC/{YEAR}/{MM}/{DD}/arome_arctic_det_2_5km_{formatted_date}T{HH}Z.nc'
            #variable='SFX_TS'
            MODdataset = ncfile(MODin, 'r')
            MODdata    = MODdataset[variable][timestep,:,:]
            MOD_times = MODdataset.variables['time'][:]
            t_unit = MODdataset.variables['time'].units
            t_cal = "proleptic_gregorian" 
            MOD_nctimes=[]; MOD_nctimes.append(num2date(MOD_times,units = t_unit,calendar = t_cal))
            MOD_nctimenum = np.squeeze(dates.date2num(MOD_nctimes))
            MOD_nctimes =np.squeeze(MOD_nctimes)   
        
        if self.model=='MF-AROME':
            fc_data_path='/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/MF-AROME/merged/'
            MODin = fc_data_path+filein
            #
            MODdataset = ncfile(MODin, 'r')
            MODdata    = MODdataset[variable][timestep,:,:]
            MOD_times = MODdataset.variables['time'][:]
            t_unit = MODdataset.variables['time'].units
            t_cal = MODdataset.variables['time'].calendar
            if "day as %Y%m%d.%f" in t_unit:
                    MOD_nctimes = [
                        datetime.strptime(str(int(time)), "%Y%m%d") + timedelta(days=time % 1)
                        for time in MOD_times
                    ]      
            #MOD_nctimes=[]; MOD_nctimes.append(num2date(MOD_times,units = t_unit,calendar = t_cal))
            MOD_nctimes   = np.squeeze(MOD_nctimes)
            MOD_nctimenum = np.squeeze(dates.date2num(MOD_nctimes))
    
        if self.model=='MF-ARPEGE':
            fc_data_path='/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/MF-ARPEGE/merged/'
            MODin = fc_data_path+filein
            #
            MODdataset = ncfile(MODin, 'r')
            MODdata    = MODdataset[variable][timestep,:,:]
            MOD_times = MODdataset.variables['time'][:]
            t_unit = MODdataset.variables['time'].units
            t_cal = MODdataset.variables['time'].calendar
            if "day as %Y%m%d.%f" in t_unit:
                    MOD_nctimes = [
                        datetime.strptime(str(int(time)), "%Y%m%d") + timedelta(days=time % 1)
                        for time in MOD_times
                    ]      
            #MOD_nctimes=[]; MOD_nctimes.append(num2date(MOD_times,units = t_unit,calendar = t_cal))
            MOD_nctimes   = np.squeeze(MOD_nctimes)
            MOD_nctimenum = np.squeeze(dates.date2num(MOD_nctimes))
            
        if self.model=='ECMWF-IFS': 
            fc_data_path='/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/ECMWF-IFS/' 
            MODin = fc_data_path+filein
            #
            MODdataset = ncfile(MODin, 'r')
            MODdata    = MODdataset[variable][timestep,:,:]
            MOD_times = MODdataset.variables['time'][:]
            t_unit = MODdataset.variables['time'].units
            t_cal = MODdataset.variables['time'].calendar
            MOD_nctimes=[]; MOD_nctimes.append(num2date(MOD_times,units = t_unit,calendar = t_cal))
            MOD_nctimes   = np.squeeze(MOD_nctimes)
            MOD_nctimenum = np.squeeze(dates.date2num(MOD_nctimes))
            
            
        if self.model=='ECMWF-AIFS':
            fc_data_path='/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/ECMWF-AIFS/'
            MODin = fc_data_path+filein
            #
            MODdataset = ncfile(MODin, 'r')
            MODdata    = MODdataset[variable][timestep,:,:]
            MOD_times = MODdataset.variables['time'][:]
            t_unit = MODdataset.variables['time'].units
            t_cal = MODdataset.variables['time'].calendar
            MOD_nctimes=[]; MOD_nctimes.append(num2date(MOD_times,units = t_unit,calendar = t_cal))
            MOD_nctimes   = np.squeeze(MOD_nctimes)
            MOD_nctimenum = np.squeeze(dates.date2num(MOD_nctimes))
            
            
        if self.model=='DWD-ICON':
            fc_data_path='/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/DWD-ICON/'
            MODin = fc_data_path+filein
            #
            MODdataset = ncfile(MODin, 'r')
            MODdata    = MODdataset[variable][timestep,:,:]
            MOD_times = MODdataset.variables['time'][:]
            t_unit = MODdataset.variables['time'].units
            t_cal = MODdataset.variables['time'].calendar
            MOD_nctimes=[]; MOD_nctimes.append(num2date(MOD_times,units = t_unit,calendar = t_cal))
            MOD_nctimes   = np.squeeze(MOD_nctimes)
            MOD_nctimenum = np.squeeze(dates.date2num(MOD_nctimes))
            
              
        MODdataset.close()
               
        self.MODnctimes   = MOD_nctimes[timestep]
        self.MODnctimenum = MOD_nctimenum[timestep]
                
        if self.model=='MET-AROMEArctic':    
            self.MODprojdata  = np.squeeze(MODdata)
        elif self.model=='MF-AROME':
            tmpdata = np.squeeze(MODdata)
            self.MODprojdata  = tmpdata[::2, ::2]
        elif self.model=='MF-ARPEGE':
            self.MODprojdata  = np.squeeze(MODdata)
        else:
            self.MODprojdata  = remapfield(MODdata.ravel(),self.idx_pos_ij)
        
        
        
    def DefineSICMask(self):
        
        date = self.MODnctimes
        
        # LOAD AMSR2 Sea-Ice Product
        fc_data_path   = '/lustre/storeB/project/fou/hi/oper/barents_eps/archive/obs/'
        file_name_stub = 'barents_icec-obs_%s' #barents_icec-obs_20240415T00Z.nc
        datetime_str = f"{date.year:04d}{date.month:02d}{date.day:02d}"
        file_prename = file_name_stub % datetime_str
        infile_name = glob.glob(fc_data_path + file_prename + '*.nc')
        
        if infile_name:
            
            IceIn = xr.open_mfdataset(infile_name)
            lats = np.array(IceIn['lat'])
            lons = np.array(IceIn['lon'])
            sicn = np.array(IceIn['ice_conc'])
            
            #sicn=np.squeeze(np.where(sicn<0.99,np.nan,sicn))  
            
            self.SIC_mask=np.squeeze(np.flipud(np.squeeze(sicn)))
            IceIn.close()
                  
        
    def RetrieveMODIS(self,masksic):
        print ('--> RetrieveMODIS')
        #MODIS grid (same as AROME Arctic)      
        modisin='/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/observations/remotesensing/aa-modis-2024.nc'
    
        modisdataset = ncfile(modisin, 'r')       
        tm = np.argmin(np.abs(self.MODnctimenum-self.modis_nctimenum))
        
        print('   Model date:',self.MODnctimes)
        print('   MODIS date:',self.modis_nctimes[tm])
            
        tempmodis = np.squeeze(modisdataset['modis_sist_lw'][tm,:,:])
        
        tempmodis = ma.filled(tempmodis, fill_value=np.nan)
        
        
        
        
        if masksic:
            print ('   Masking sic < 0.2')
            tempmodis = np.squeeze(np.where(self.SIC_mask<0.01,np.nan,tempmodis.data))
          #  tempmodis = np.squeeze(np.where(self.SIC_mask==np.nan,np.nan,tempmodis.data))
                        
        self.modisdata = tempmodis
        
        modisdataset.close()
        
    def ConditionalEvaluateModel(self,cond,cmin,cmax,cinc):
        
        if cond=='SIC':
            condfield = self.SIC_mask
            bins = np.arange(cmin, cmax + cinc, cinc) 
        elif cond=='TS':
            condfield = self.modisdata-273.15
            bins = np.arange(cmin, cmax + cinc, cinc)
            
        modfield=self.MODprojdata
        obsfield=self.modisdata
        
        bias_list=[];mse_list=[]
        # Iterate over the bins
        for i in range(len(bins) - 1):
            # Define the condition range
            lower_bound = bins[i]
            upper_bound = bins[i + 1]
        
            # Create a mask for elements within the current condition range
            condition_mask = (condfield >= lower_bound) & (condfield < upper_bound)
        
            # Extract corresponding values from modfield and obsfield
            mod_vals = modfield[condition_mask]
            obs_vals = obsfield[condition_mask]
        
            # Ensure there is data in this range before calculating
            if (len(mod_vals) > 0) and (len(obs_vals) > 0) and not np.isnan(np.nanmean(obs_vals-mod_vals)) and not np.isnan(np.nanmean(mod_vals)):
                # Calculate the bias and RMSE for this condition range
                #print ('mod mean',np.nanmean(mod_vals))
                #print ('obs mean',np.nanmean(obs_vals))
                
                bias = np.nanmean(mod_vals - obs_vals)  # Bias: mean(mod - obs)
                mse = np.nanmean((mod_vals - obs_vals) ** 2)  # MSE: mean((mod - obs)^2)
            
                # Append the results
                bias_list.append(bias)
                mse_list.append(mse)
            else:
                # If no data for this condition, append NaN
                bias_list.append(np.nan)
                mse_list.append(np.nan)
    
            # Return the results as a dictionary
        print("bias", bias_list,"mse", mse_list,"bins", bins[:-1])  # To show which bin corresponds to each result
        self.bias = np.array(bias_list)
        self.mse  = np.array(mse_list)     
        self.bins = np.array(bins)


In [129]:
def conditionalEvaluation(start_date,end_date,start_leadtime,end_leadtime,modeleval,condvar,cmin,cmax,cinc):


    m1 = SpatialEvaluation(modeleval=modeleval)
    m1.defineGridTransforms()

    # Loop over the days from start to end
    current_date = start_date
    first=1
    while current_date <= end_date:
        # Format the date for the filename (e.g., 20240410)
        formatted_date = current_date.strftime('%Y%m%d')
        YEAR=current_date.strftime('%Y')
        MM  =current_date.strftime('%m')
        DD  =current_date.strftime('%d')
        for HH in ['00','12']:   
        
            if modeleval=='MET-AROMEArctic':
                idir='/lustre/storeB/project/nwp/SALIENSEAS/SvalMIZ2024/models/'
                filein = f'AROME_ARCTIC/{YEAR}/{MM}/{DD}/arome_arctic_det_2_5km_{formatted_date}T{HH}Z.nc'
                variable='SFX_TS'
            if modeleval=='MF-AROME':
                filein = f'AROME_SVALBARD_{formatted_date}{HH}00.nc'
                variable='TS'
            if modeleval=='MF-ARPEGE':
                filein = f'ARPEGE_SVALBARD_{formatted_date}{HH}00.nc'
                variable='TS'
            if modeleval=='ECMWF-IFS':     
                filein = f'Arctic.ECMWF_extracted_{formatted_date}T{HH}Z.nc'
                variable='SKT'
            if modeleval=='ECMWF-AIFS':
                filein = f'Arctic.ECMWF_AIFS_extracted_{formatted_date}T{HH}Z.nc'
                variable='skt'
            if modeleval=='DWD-ICON':
                filein = f'region.icon_global_{formatted_date}{HH}_T_ICE.nc'
                variable='ist'
            
            
            print ('Input File: ',filein)
            
            for ti in range(start_leadtime,end_leadtime+1):
    
                # Call the ModelTransform method with the current date's filename
                m1.ModelTransform(filein=filein, timestep=ti, variable=variable)
                m1.DefineSICMask()
                m1.RetrieveMODIS(masksic=True)
                m1.ConditionalEvaluateModel(cond=condvar,cmin=cmin ,cmax=cmax,cinc=cinc)
                if first==1:
                    bias=m1.bias.reshape(len(m1.bins)-1, 1)
                    mse=m1.mse.reshape(len(m1.bins)-1, 1)
                    first=0
                else:
                    bias=np.hstack((bias,m1.bias.reshape(len(m1.bins)-1, 1)))
                    mse=np.hstack((mse,m1.mse.reshape(len(m1.bins)-1, 1)))
                
            

        # Move to the next day
        current_date += timedelta(days=1)
    
    rmse_values = np.sqrt(np.nanmean(mse,axis=1))
    bias_values = np.nanmean(bias,axis=1)
    return rmse_values,bias_values


In [128]:
rmse_values

[]

### 1 - Evaluation, conditional = SIC, leadtime 13-24h 

In [ ]:
# Define start and end date
start_date = datetime(2024, 4, 6)  # 
end_date = datetime(2024, 4, 30)    # 

pls=13;ple=25
als=2 ;ale=5
condvar = 'SIC'

models = ['ECMWF-IFS', 'ECMWF-AIFS', 'DWD-ICON', 'MET-AROMEArctic', 'MF-AROME', 'MF-ARPEGE']
rmse_values=[]; bias_values=[]
#models = ['MF-AROME']

for model in models:

    if model=='ECMWF-AIFS':
        ls=als;  le=ale
    else:
        ls=pls;  le=ple
    
    # Run conditional evaluations for each model and store results
    rmse, bias = conditionalEvaluation(start_date, end_date, ls, le, model, condvar, 0., 1., 0.2)
    rmse_values.append(rmse1)
    bias_values.append(bias1)


# Create a DataFrame to organize the results
results_df = pd.DataFrame({
    'Model': models,
    'RMSE': rmse_values,
    'Bias': bias_values
})

# Save the results to a CSV file
results_df.to_csv('MODISeval_'+condvar+'_'+str(pls)+'-'+str(ple)+'.csv', index=False)

print("Results saved ...")



Input File:  Arctic.ECMWF_extracted_20240406T00Z.nc
--> RetrieveMODIS
   Model date: 2024-04-06 13:00:00
   MODIS date: 2024-04-06 13:00:00
   Masking sic < 0.2
bias [-0.5122166613695915, -0.17120636941250023, 0.5850665608028381, 1.1677296446499221, 2.410189244322151] mse [10.988180438696185, 13.615706711450972, 24.93166827402003, 20.681728946714298, 15.626637050295928] bins [0.  0.2 0.4 0.6 0.8]
--> RetrieveMODIS
   Model date: 2024-04-06 14:00:00
   MODIS date: 2024-04-06 14:00:00
   Masking sic < 0.2
bias [-0.12468582334425551, 0.1523966887305458, 1.3004334696812632, 2.73789768321622, 2.908974222790016] mse [13.996781499059166, 16.006494495339727, 30.27444600289176, 37.9238964319931, 20.922206037895858] bins [0.  0.2 0.4 0.6 0.8]
--> RetrieveMODIS
   Model date: 2024-04-06 15:00:00
   MODIS date: 2024-04-06 15:00:00
   Masking sic < 0.2
bias [nan, nan, nan, nan, nan] mse [nan, nan, nan, nan, nan] bins [0.  0.2 0.4 0.6 0.8]
--> RetrieveMODIS
   Model date: 2024-04-06 16:00:00
   MODI

--> RetrieveMODIS
   Model date: 2024-04-07 14:00:00
   MODIS date: 2024-04-07 14:00:00
   Masking sic < 0.2
bias [nan, nan, nan, nan, nan] mse [nan, nan, nan, nan, nan] bins [0.  0.2 0.4 0.6 0.8]
--> RetrieveMODIS
   Model date: 2024-04-07 15:00:00
   MODIS date: 2024-04-07 15:00:00
   Masking sic < 0.2
bias [6.641922759554185, 4.82571686240301, 3.4254401172972995, 4.1038654727533865, 3.270700871694695] mse [128.812163853187, 97.58759397396491, 79.32304936743866, 63.5906416272142, 26.14077743232329] bins [0.  0.2 0.4 0.6 0.8]
--> RetrieveMODIS
   Model date: 2024-04-07 16:00:00
   MODIS date: 2024-04-07 16:00:00
   Masking sic < 0.2
bias [nan, nan, nan, nan, nan] mse [nan, nan, nan, nan, nan] bins [0.  0.2 0.4 0.6 0.8]
--> RetrieveMODIS
   Model date: 2024-04-07 17:00:00
   MODIS date: 2024-04-07 17:00:00
   Masking sic < 0.2
bias [8.306264163292687, 5.767886367534437, 4.459844997219059, 4.825876639866034, 3.0817152001425185] mse [136.50779194494106, 97.50748756138631, 80.917692395099

--> RetrieveMODIS
   Model date: 2024-04-08 15:00:00
   MODIS date: 2024-04-08 15:00:00
   Masking sic < 0.2
bias [0.9001853633974872, 0.5051933906323683, 0.9837479503726292, 0.9153746714919287, 1.7139440775401333] mse [1.2015906405926167, 0.497749384175141, 2.389369863748347, 3.9151118447636217, 6.679047497681315] bins [0.  0.2 0.4 0.6 0.8]
--> RetrieveMODIS
   Model date: 2024-04-08 16:00:00
   MODIS date: 2024-04-08 16:00:00
   Masking sic < 0.2
bias [3.659731065799558, 2.494481368885056, 2.7230100158846886, 2.5072345165899965, 2.154423548920423] mse [35.37793747942232, 27.796371475813412, 27.70383324054716, 24.926742808081745, 13.587579392533657] bins [0.  0.2 0.4 0.6 0.8]
--> RetrieveMODIS
   Model date: 2024-04-08 17:00:00
   MODIS date: 2024-04-08 17:00:00
   Masking sic < 0.2
bias [6.5069086529083915, 3.9643731194963174, 3.3303463590713216, 2.987140751252832, 2.703159434969885] mse [83.06774512530738, 40.709507362107004, 32.86498050778113, 28.041713419774585, 17.388438361351263

--> RetrieveMODIS
   Model date: 2024-04-09 15:00:00
   MODIS date: 2024-04-09 15:00:00
   Masking sic < 0.2
bias [1.1411762441716145, 1.1098434459978124, 0.9895498239671855, 1.0421102453080877, 2.2654254065561337] mse [4.5241315892815495, 9.175942750064817, 13.13364688444436, 15.81841109844811, 13.08917446990089] bins [0.  0.2 0.4 0.6 0.8]
--> RetrieveMODIS
   Model date: 2024-04-09 16:00:00
   MODIS date: 2024-04-09 16:00:00
   Masking sic < 0.2
bias [2.2603641248623534, 2.553139409368849, 2.252304264841695, 2.248639291144154, 2.702906978495586] mse [20.843060978901747, 25.614699048602727, 24.636961793738497, 26.413382454751154, 16.07598380707581] bins [0.  0.2 0.4 0.6 0.8]
--> RetrieveMODIS
   Model date: 2024-04-09 17:00:00
   MODIS date: 2024-04-09 17:00:00
   Masking sic < 0.2
bias [nan, nan, nan, nan, nan] mse [nan, nan, nan, nan, nan] bins [0.  0.2 0.4 0.6 0.8]
--> RetrieveMODIS
   Model date: 2024-04-09 18:00:00
   MODIS date: 2024-04-09 18:00:00
   Masking sic < 0.2
bias [3.1

--> RetrieveMODIS
   Model date: 2024-04-10 16:00:00
   MODIS date: 2024-04-10 16:00:00
   Masking sic < 0.2
bias [nan, nan, nan, nan, nan] mse [nan, nan, nan, nan, nan] bins [0.  0.2 0.4 0.6 0.8]
--> RetrieveMODIS
   Model date: 2024-04-10 17:00:00
   MODIS date: 2024-04-10 17:00:00
   Masking sic < 0.2
bias [1.155468811349806, 2.2320313206772533, 3.05762333189459, 3.5346918726268286, 2.532889518190733] mse [17.14565249268211, 31.28528690192221, 44.560050569993905, 47.880116088884215, 20.36464770392714] bins [0.  0.2 0.4 0.6 0.8]
--> RetrieveMODIS
   Model date: 2024-04-10 18:00:00
   MODIS date: 2024-04-10 18:00:00
   Masking sic < 0.2
bias [nan, nan, nan, nan, 6.140630243354586] mse [nan, nan, nan, nan, 37.877706943552695] bins [0.  0.2 0.4 0.6 0.8]
--> RetrieveMODIS
   Model date: 2024-04-10 19:00:00
   MODIS date: 2024-04-10 19:00:00
   Masking sic < 0.2
bias [6.8190622120330335, 5.318930692447337, 5.174197731231103, 3.9471540926074633, 2.493669443524902] mse [155.3685527193293, 1

...

--> RetrieveMODIS
   Model date: 2024-04-17 21:00:00
   MODIS date: 2024-04-17 21:00:00
   Masking sic < 0.2
bias [nan, nan, nan, nan, nan] mse [nan, nan, nan, nan, nan] bins [0.  0.2 0.4 0.6 0.8]
--> RetrieveMODIS
   Model date: 2024-04-17 22:00:00
   MODIS date: 2024-04-17 22:00:00
   Masking sic < 0.2
bias [14.61849827695074, 12.984783206373768, 10.368422118730432, 5.361411909456379, 5.4926680755176145] mse [295.75044001174365, 233.91365576495747, 164.64819069694838, 91.3726044220713, 54.306947091295385] bins [0.  0.2 0.4 0.6 0.8]
--> RetrieveMODIS
   Model date: 2024-04-17 23:00:00
   MODIS date: 2024-04-17 23:00:00
   Masking sic < 0.2
bias [4.618987983170017, 3.676805515712405, 1.803431976874418, 1.4385719074191603, 2.4350416571453577] mse [44.83818716836744, 33.361561204476, 21.354928353835678, 21.50816382268335, 22.95969147501456] bins [0.  0.2 0.4 0.6 0.8]
--> RetrieveMODIS
   Model date: 2024-04-18 00:00:00
   MODIS date: 2024-04-18 00:00:00
   Masking sic < 0.2
bias [nan, na

--> RetrieveMODIS
   Model date: 2024-04-18 22:00:00
   MODIS date: 2024-04-18 22:00:00
   Masking sic < 0.2
bias [16.581714136062725, 16.270422591753192, 14.690145802438444, 10.821461993901545, 7.422166287303668] mse [299.9861348734773, 294.8410392949591, 247.72720640592726, 150.74478256070418, 70.11208216828948] bins [0.  0.2 0.4 0.6 0.8]
--> RetrieveMODIS
   Model date: 2024-04-18 23:00:00
   MODIS date: 2024-04-18 23:00:00
   Masking sic < 0.2
bias [nan, nan, nan, nan, 4.2119153771635265] mse [nan, nan, nan, nan, 22.08458456902686] bins [0.  0.2 0.4 0.6 0.8]
--> RetrieveMODIS
   Model date: 2024-04-19 00:00:00
   MODIS date: 2024-04-19 00:00:00
   Masking sic < 0.2
bias [1.9644667946148493, 1.9616516889577675, -0.02021057472162614, 0.3241931193939531, 1.2736692612946825] mse [16.19670019270072, 15.535440926049489, 17.402569152207537, 14.45510658663531, 18.53710894154112] bins [0.  0.2 0.4 0.6 0.8]
--> RetrieveMODIS
   Model date: 2024-04-19 01:00:00
   MODIS date: 2024-04-19 01:00:

--> RetrieveMODIS
   Model date: 2024-04-20 00:00:00
   MODIS date: 2024-04-20 00:00:00
   Masking sic < 0.2
bias [nan, nan, nan, nan, nan] mse [nan, nan, nan, nan, nan] bins [0.  0.2 0.4 0.6 0.8]
--> RetrieveMODIS
   Model date: 2024-04-20 01:00:00
   MODIS date: 2024-04-20 01:00:00
   Masking sic < 0.2
bias [3.7193830993283648, 4.178274020551924, 1.220533527484637, 1.7351156762980675, 1.4588504601109864] mse [67.65497762221888, 83.9758711569612, 42.74484272305472, 28.666231660863687, 23.25053222440554] bins [0.  0.2 0.4 0.6 0.8]
Input File:  Arctic.ECMWF_extracted_20240419T12Z.nc
--> RetrieveMODIS
   Model date: 2024-04-20 01:00:00
   MODIS date: 2024-04-20 01:00:00
   Masking sic < 0.2
bias [3.844291843114575, 4.267504505747996, 1.2436014257051229, 1.894807898182748, 1.7135866336441274] mse [67.53273791602976, 81.2269617788915, 41.791929813410036, 28.463198758154714, 23.151370752707045] bins [0.  0.2 0.4 0.6 0.8]
--> RetrieveMODIS
   Model date: 2024-04-20 02:00:00
   MODIS date: 20

--> RetrieveMODIS
   Model date: 2024-04-21 01:00:00
   MODIS date: 2024-04-21 01:00:00
   Masking sic < 0.2
bias [3.8241897777637526, 2.1715967802206486, 0.2149844633679248, -0.502513156410606, 2.3981663409805614] mse [71.05424519051465, 64.82293076931235, 58.46116960034878, 36.49217672276092, 37.019057487186096] bins [0.  0.2 0.4 0.6 0.8]
Input File:  Arctic.ECMWF_extracted_20240420T12Z.nc
--> RetrieveMODIS
   Model date: 2024-04-21 01:00:00
   MODIS date: 2024-04-21 01:00:00
   Masking sic < 0.2
bias [3.552701211622337, 1.9087100991254022, -0.08227363000552881, -0.5923021973677449, 2.3303917995529146] mse [68.34495194201779, 63.81526871097578, 58.02919056968156, 35.788643140051235, 36.18328257840343] bins [0.  0.2 0.4 0.6 0.8]
--> RetrieveMODIS
   Model date: 2024-04-21 02:00:00
   MODIS date: 2024-04-21 02:00:00
   Masking sic < 0.2
bias [nan, nan, nan, nan, nan] mse [nan, nan, nan, nan, nan] bins [0.  0.2 0.4 0.6 0.8]
--> RetrieveMODIS
   Model date: 2024-04-21 03:00:00
   MODIS d

--> RetrieveMODIS
   Model date: 2024-04-22 01:00:00
   MODIS date: 2024-04-22 01:00:00
   Masking sic < 0.2
bias [nan, nan, nan, nan, 3.4657843993237094] mse [nan, nan, nan, nan, 12.027312757313794] bins [0.  0.2 0.4 0.6 0.8]
Input File:  Arctic.ECMWF_extracted_20240421T12Z.nc
--> RetrieveMODIS
   Model date: 2024-04-22 01:00:00
   MODIS date: 2024-04-22 01:00:00
   Masking sic < 0.2
bias [nan, nan, nan, nan, 3.6591418669750766] mse [nan, nan, nan, nan, 13.399586090921055] bins [0.  0.2 0.4 0.6 0.8]
--> RetrieveMODIS
   Model date: 2024-04-22 02:00:00
   MODIS date: 2024-04-22 02:00:00
   Masking sic < 0.2
bias [6.424379372430596, 5.180270682262701, 1.5059561377931705, 2.1846621911696777, 6.804864603619345] mse [125.36448357661297, 115.06962271447793, 53.93230953748708, 61.150038604720024, 119.32053176348766] bins [0.  0.2 0.4 0.6 0.8]
--> RetrieveMODIS
   Model date: 2024-04-22 03:00:00
   MODIS date: 2024-04-22 03:00:00
   Masking sic < 0.2
bias [nan, nan, nan, nan, nan] mse [nan, n

#### Some test environment